# 【notebook｜N-21】NumPy 与张量思维
> 对应理论篇：`../../../v1.0/00-预备/PY4-NumPy与张量思维.md`（doccode = PY4）
> 对应代码：`../../../v1.0/00-预备/code/py4_numpy.py`（filekey = F-pre-py4）
> 前置：PY1/PY2 的 Python 基础；本章不讲数学，只讲形状、内存与执行语义。
> **Colab**：CPU 运行时，全部运行约 10 秒。

## 学习目标
1. 用 shape/dtype/strides 描述 ndarray，并分清视图与拷贝；
2. 掌握基本/花式/布尔索引与赋值语义；
3. 掌握广播严格规则并会诊断 shape 报错；
4. 把朴素循环改写成 matmul/广播，理解 axis 归约与 keepdims；
5. 用 Generator 管理随机数与种子，认识 linalg 常用函数。


## 1. ndarray 与内存布局（【定义 PY4.1.1】～【代码 PY4.1.7】）

In [ ]:
# 【N-21.c1】
import numpy as np
print('numpy', np.__version__)
a = np.arange(12, dtype=np.float32).reshape(3, 4)
print('shape', a.shape, '| dtype', a.dtype, '| strides', a.strides, '| nbytes', a.nbytes)

In [ ]:
# 【N-21.c2】
b = a.T                            # 转置：视图
c = a[1:, ::2]                     # 切片：视图
d = a[[0, 2]]                      # 花式索引：拷贝
print('a.T strides', b.strides, '| shares_memory', np.shares_memory(a, b))
print('slice base is a:', c.base is a, '| fancy base is None:', d.base is None)
a[0, 0] = 99
print('视图看到原内存修改 b[-1,0] =', b[-1, 0])

## 2. 索引与赋值（【定义 PY4.2.1】～【注 PY4.2.7】）

In [ ]:
# 【N-21.c3】
m = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print('基本索引:', m[1, 2], '| 切片视图:\n', m[:2, :2])
print('花式索引 [0,2] 行:\n', m[[0, 2]])
print('布尔掩码 >5:', m[m > 5])
v = m[0]; v[:] = -1                # 视图原地写回
m[m < 0] = 0                       # 掩码赋值：原地
print('写回后:\n', m)

## 3. 广播严格规则（【定义 PY4.3.1】～【注 PY4.3.6】）

In [ ]:
# 【N-21.c4】
x = np.zeros((3, 4))
print((x + np.arange(4)).shape)             # (3,4)
print((x + np.arange(3)[:, None]).shape)    # (3,4)，None 插轴
y = np.broadcast_to(np.arange(4), (3, 4))
print('broadcast_to strides', y.strides)    # 扩展维 stride=0
try:
    x + np.arange(3)                        # (3,4) vs (3,) 冲突
except ValueError as e:
    print('broadcast error:', str(e)[:70])

## 4. 向量化：朴素循环 -> matmul（【算法 PY4.4.2】～【注 PY4.4.6】）

In [ ]:
# 【N-21.c5】
import time
rng = np.random.default_rng(0)
X = rng.standard_normal((32, 64)); W = rng.standard_normal((32, 64)); b = rng.standard_normal(32)

def matmul_naive(X, W, b):
    B, D = X.shape; M = W.shape[0]
    Z = np.empty((B, M))
    for i in range(B):
        for j in range(M):
            s = 0.0
            for kk in range(D):
                s += X[i, kk] * W[j, kk]
            Z[i, j] = s + b[j]
    return Z

t0 = time.perf_counter(); Zn = matmul_naive(X, W, b); tn = time.perf_counter() - t0
t0 = time.perf_counter(); Zv = X @ W.T + b; tv = time.perf_counter() - t0
print('allclose', np.allclose(Zn, Zv), '| naive %.1f ms' % (tn * 1e3), '| vectorized %.1f us' % (tv * 1e6))

In [ ]:
# 【N-21.c6】
Ze = np.einsum('bd,md->bm', X, W) + b
print('einsum 与 @ 一致:', np.allclose(Ze, X @ W.T + b))

## 5. 归约、axis 与 keepdims（【定义 PY4.5.1】～【注 PY4.5.5】）

In [ ]:
# 【N-21.c7】
x = np.arange(12, dtype=np.float64).reshape(3, 4)
print(x.sum(axis=0).shape, x.sum(axis=1).shape, x.sum().shape)   # (4,) (3,) ()
print(x.max(axis=1, keepdims=True).shape)                        # (3,1)
print(np.allclose(x - x.mean(axis=1, keepdims=True), x - x.mean(axis=1)[:, None]))

def softmax_np(z, axis=-1):
    m = z.max(axis=axis, keepdims=True)
    e = np.exp(z - m)
    return e / e.sum(axis=axis, keepdims=True)

p = softmax_np(np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]]))
print('每行和:', p.sum(axis=-1))

## 6. 随机数：Generator 与种子（【注 PY4.6.1】～【代码 PY4.6.5】）

In [ ]:
# 【N-21.c8】
rng = np.random.default_rng(0)
print(rng.standard_normal(3)[:2], rng.integers(0, 10, size=3))
print('同种子可复现:', np.allclose(np.random.default_rng(0).standard_normal(3), np.random.default_rng(0).standard_normal(3)))
children = np.random.SeedSequence(1234).spawn(2)
print('并行子流:', [np.random.default_rng(c).integers(0, 100, size=2).tolist() for c in children])

## 7. 线性代数：matmul 与 linalg（【定义 PY4.7.1】～【注 PY4.7.5】）

In [ ]:
# 【N-21.c9】
A = rng.standard_normal((3, 3)) + 3 * np.eye(3)
rhs = rng.standard_normal(3)
sol = np.linalg.solve(A, rhs)
print('solve 残差:', np.linalg.norm(A @ sol - rhs))
print('批量 matmul:', (rng.standard_normal((8, 2, 3)) @ rng.standard_normal((3, 5))).shape)  # (8,2,5)
print('norm:', np.linalg.norm(sol), '| svd 形状:', [s.shape for s in np.linalg.svd(A)[:2]])

## 8. 小结
1. 形状即类型：先推 shape，再写算子。
2. 视图共享内存，花式/布尔索引返回拷贝。
3. 广播 = 右对齐 + 1 可扩 + 缺失补 1；归约用 keepdims 才能广播回去。
4. 用 `@`/`einsum` 表达循环，速度来自 BLAS 与 C 内核。

**练习**（见理论篇【练习 PY4.9.1】～【练习 PY4.9.3】）。